# Большое домашнее задание 2

Александр Степаненко 231 

In [ ]:
pip install sacrebleu

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

#import torchtext

import spacy
import pandas as pd
import numpy as np
import sentencepiece as spm
from tqdm.notebook import tqdm
import wandb
from comet_ml import Experiment
import sacrebleu
import math

from collections import Counter
import matplotlib.pyplot as plt

### Data

In [ ]:
#!git clone https://github.com/k1ngofdarks/DL-Transformers.git

In [ ]:
def load_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
    return lines

try:
    DATA_FOLDER = "DL-Transformers/bhw2-data/data"
    train_de = load_file(DATA_FOLDER + "/train.de-en.de")
    train_en = load_file(DATA_FOLDER + "/train.de-en.en")
except:
    DATA_FOLDER = "bhw2-data/data"
    train_de = load_file(DATA_FOLDER + "/train.de-en.de")
    train_en = load_file(DATA_FOLDER + "/train.de-en.en")

val_de = load_file(DATA_FOLDER + "/val.de-en.de")
val_en = load_file(DATA_FOLDER + "/val.de-en.en")

test_de = load_file(DATA_FOLDER + "/test1.de-en.de")

print("Train size:", len(train_de))
print("Val   size:", len(val_de))
print("Test  size:", len(test_de))

print(DATA_FOLDER)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    print("AMP dtype: bfloat16 (GradScaler disabled)")
elif DEVICE.type == "cuda":
    amp_dtype = torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=True, init_scale=2**10, growth_interval=2000)
    print("AMP dtype: float16 (GradScaler enabled)")
else:
    amp_dtype = torch.float32
    scaler = torch.amp.GradScaler("cpu", enabled=False)
    print("AMP disabled for CPU")


# Preprocessing

In [ ]:
from collections import Counter

class Tokenizer:
    def __init__(self, max_size=20000, min_freq=1):
        self.max_size = max_size
        self.min_freq = min_freq
        self.special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
        
        self.word2id = {}
        self.id2word = {}

    def build_vocab(self, sentences):
        counter = Counter()
        for s in sentences:
            counter.update(s.split())

        words = [w for w, c in counter.items() if c >= self.min_freq]
        words = sorted(words, key=lambda w: counter[w], reverse=True)
        words = words[:self.max_size]

        vocab_words = self.special_tokens + words

        self.word2id = {w: i for i, w in enumerate(vocab_words)}
        self.id2word = {i: w for w, i in self.word2id.items()}

    def encode(self, sentence):
        tokens = sentence.split()
        ids = [self.word2id.get(w, self.word2id["<unk>"]) for w in tokens]
        return [self.word2id["<bos>"]] + ids + [self.word2id["<eos>"]]

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.id2word.get(i, "<unk>")
            if w in ["<bos>", "<pad>"]:
                continue
            if w == "<eos>":
                break
            words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2id)

In [ ]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_tokenizer, tgt_tokenizer, max_len=80):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_ids = self.src_tokenizer.encode(self.src_sentences[idx])[:self.max_len]
        tgt_ids = self.tgt_tokenizer.encode(self.tgt_sentences[idx])[:self.max_len]
        
        return torch.tensor(src_ids), torch.tensor(tgt_ids)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch, pad_id_src, pad_id_tgt):
    src_batch, tgt_batch = zip(*batch)

    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=pad_id_src)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=pad_id_tgt)

    return src_padded, tgt_padded

In [ ]:
from torch.utils.data import DataLoader

de_tokenizer = Tokenizer(max_size=20000, min_freq=1)
en_tokenizer = Tokenizer(max_size=20000, min_freq=1)

de_tokenizer.build_vocab(train_de)
en_tokenizer.build_vocab(train_en)

train_dataset = TranslationDataset(train_de, train_en, de_tokenizer, en_tokenizer, max_len=80)
val_dataset = TranslationDataset(val_de, val_en, de_tokenizer, en_tokenizer, max_len=80)

BATCH = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_fn(
        batch,
        pad_id_src=de_tokenizer.word2id["<pad>"],
        pad_id_tgt=en_tokenizer.word2id["<pad>"]
    )
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False,
    collate_fn=lambda batch: collate_fn(
        batch,
        pad_id_src=de_tokenizer.word2id["<pad>"],
        pad_id_tgt=en_tokenizer.word2id["<pad>"]
    )
)

## Модули

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.embed_dim = embed_dim

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.embed_dim)
    
    
class PositionalEncoder(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        pos_features = torch.zeros(max_length, embed_dim)
        positions = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        freqs = torch.exp(torch.arange(0, embed_dim, 2, dtype=torch.float) * 
                          (-math.log(10000.0) / embed_dim)).unsqueeze(0)

        arguments = positions * freqs
        pos_features[:, 0::2] = torch.sin(arguments)
        pos_features[:, 1::2] = torch.cos(arguments)
        pos_features = pos_features.unsqueeze(0)

        self.register_buffer("pos_features", pos_features)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pos_features[:, :seq_len, :]
        return self.dropout(x)

In [ ]:
class TransformerModel(nn.Module):
    def __init__(
        self,
        src_vocab,
        tgt_vocab,
        d_model=384,
        nhead=6,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=1024,
        dropout=0.1,
        max_len=100
    ):
        super().__init__()
        self.src_embed = TokenEmbedding(src_vocab, d_model, pad_id=de_tokenizer.word2id["<pad>"])
        self.tgt_embed = TokenEmbedding(tgt_vocab, d_model, pad_id=en_tokenizer.word2id["<pad>"])
        self.pos_enc = PositionalEncoder(max_len, d_model, dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab)
    
    def forward(self, src, tgt, pad_id_src, pad_id_tgt):
        src_key_padding_mask = (src == pad_id_src)
        tgt_key_padding_mask = (tgt == pad_id_tgt)
        tgt_len = tgt.size(1)
        
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(tgt.device)

        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return self.fc_out(out)

## Train

In [ ]:
def _unwrap_model(m):
    return m.module if hasattr(m, "module") else m

In [ ]:
def greedy_decode(model, src, src_key_padding_mask, max_len, bos_id, eos_id):
    model_for_decode = _unwrap_model(model)
    device = src.device
    batch_size = src.size(0)

    # 1. Сначала эмбеддинги для энкодера!
    src_emb = model_for_decode.pos_enc(model_for_decode.src_embed(src))
    memory = model_for_decode.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    ys = torch.full((batch_size, 1), bos_id, dtype=torch.long, device=device)

    for _ in range(max_len - 1):
        tgt_len = ys.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(ys.device)

        tgt_emb = model_for_decode.pos_enc(model_for_decode.tgt_embed(ys))
        
        out = model_for_decode.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        
        logits = model_for_decode.fc_out(out)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)

        if (next_token.squeeze(1) == eos_id).all():
            break

    return ys

In [ ]:
def batch_translate(model, texts, src_tokenizer, tgt_tokenizer, device, batch_size=32, max_len=80):
    model.eval()
    all_predictions = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        
        # Кодируем весь батч
        input_ids = [src_tokenizer.encode(t) for t in batch_texts]
        src_tensor = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(ids[:max_len]) for ids in input_ids], 
            batch_first=True, 
            padding_value=src_tokenizer.word2id["<pad>"]
        ).to(device)
        
        src_mask = (src_tensor == src_tokenizer.word2id["<pad>"])

        with torch.no_grad():
            out_ids = greedy_decode(
                model, src_tensor, src_mask, max_len,
                bos_id=tgt_tokenizer.word2id["<bos>"],
                eos_id=tgt_tokenizer.word2id["<eos>"]
            )

        for ids in out_ids:
            decoded = tgt_tokenizer.decode(ids.tolist())
            all_predictions.append(decoded)
            
    return all_predictions

In [ ]:
COMET_API_KEY = "1kc44ZRsKlVzok3SMdIgY8E6v"
COMET_PROJECT_NAME = "DL-Transformers-de-en"
COMET_WORKSPACE = None

def init_comet_experiment(api_key, project_name=COMET_PROJECT_NAME, workspace=COMET_WORKSPACE):
    if not api_key or api_key == "1kc44ZRsKlVzok3SMdIgY8E6v":
        print("Comet disabled: set COMET_API_KEY to enable logging")
        return None

    experiment = Experiment(
        api_key=api_key,
        project_name=project_name,
        workspace=workspace,
        auto_param_logging=False,
        auto_metric_logging=False,
    )
    print("Comet experiment initialized")
    return experiment

comet_experiment = init_comet_experiment(COMET_API_KEY)


In [ ]:
from tqdm.auto import tqdm
from IPython.display import clear_output
import copy

def train_model(model, train_loader, val_loader, src_tokenizer, tgt_tokenizer, val_src_texts, val_tgt_texts, epochs=10, device=DEVICE, comet_experiment=None):
    pad_id_src = src_tokenizer.word2id["<pad>"]
    pad_id_tgt = tgt_tokenizer.word2id["<pad>"]

    base_lr = 3e-4
    optimizer = optim.AdamW(model.parameters(), lr=base_lr, betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)

    # Warmup + cosine decay по шагам (стабильнее для Transformer, чем шаг по эпохам)
    total_steps = epochs * len(train_loader)
    warmup_steps = max(1, int(0.1 * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        min_lr_ratio = 0.02
        return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id_tgt, label_smoothing=0.1)

    if comet_experiment is not None:
        comet_experiment.log_parameters({
            "epochs": epochs,
            "base_lr": base_lr,
            "warmup_steps": warmup_steps,
            "optimizer": "AdamW",
            "weight_decay": 1e-4,
            "label_smoothing": 0.1,
            "batch_size": getattr(train_loader, "batch_size", None),
            "amp_dtype": str(amp_dtype),
        })

    history_train_loss = []
    history_val_loss = []
    history_bleu = []

    global_step = 0
    best_bleu = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        valid_updates = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - train", leave=False)
        for src, tgt in pbar:
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_expected = tgt[:, 1:]

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type == "cuda")):
                logits = model(src, tgt_input, pad_id_src=pad_id_src, pad_id_tgt=pad_id_tgt)
                loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_expected.reshape(-1))

            if not torch.isfinite(loss):
                print(f"[WARN] Non-finite loss at step {global_step}: {loss.item()}. Skipping batch.")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            if not torch.isfinite(grad_norm):
                print(f"[WARN] Non-finite grad norm at step {global_step}. Skipping optimizer step.")
                optimizer.zero_grad(set_to_none=True)
                continue

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            global_step += 1
            epoch_loss += loss.item()
            valid_updates += 1

            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.7f}", gnorm=f"{float(grad_norm):.2f}")

        train_loss = epoch_loss / max(1, valid_updates)
        history_train_loss.append(train_loss)

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for src, tgt in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - val", leave=False):
                src = src.to(device)
                tgt = tgt.to(device)

                tgt_input = tgt[:, :-1]
                tgt_expected = tgt[:, 1:]

                with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type == "cuda")):
                    logits = model(src, tgt_input, pad_id_src=pad_id_src, pad_id_tgt=pad_id_tgt)
                    loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_expected.reshape(-1))

                val_loss += loss.item()

        val_loss = val_loss / len(val_loader)
        history_val_loss.append(val_loss)

        # quick BLEU on subset (200)
        predictions = batch_translate(model, val_src_texts[:200], src_tokenizer, tgt_tokenizer, device)
        bleu = sacrebleu.corpus_bleu(predictions, [val_tgt_texts[:200]]).score
        history_bleu.append(bleu)

        if bleu > best_bleu:
            best_bleu = bleu
            best_state = copy.deepcopy(_unwrap_model(model).state_dict())

        if comet_experiment is not None:
            comet_experiment.log_metrics({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_bleu": bleu,
            }, step=epoch)

        clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(15, 5))

        ax[0].plot(history_train_loss, label='Train Loss', color='blue', marker='o')
        ax[0].plot(history_val_loss, label='Val Loss', color='orange', marker='o')
        ax[0].set_title('Loss')
        ax[0].set_xlabel('Epoch')
        ax[0].set_ylabel('Loss')
        ax[0].legend()
        ax[0].grid()

        ax[1].plot(history_bleu, label='Val BLEU', color='green', marker='o')
        ax[1].set_title('BLEU')
        ax[1].set_xlabel('Epoch')
        ax[1].set_ylabel('Score')
        ax[1].legend()
        ax[1].grid()

        plt.show()
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val BLEU: {bleu:.2f} | Best BLEU: {best_bleu:.2f}")

        if comet_experiment is not None and epoch % 3 == 0:
            torch.save(model.state_dict(), f"model_{epoch}.pth")
            comet_experiment.log_model("my_model", f"model_{epoch}.pth")

    if best_state is not None:
        _unwrap_model(model).load_state_dict(best_state)
        print(f"Loaded best checkpoint with BLEU={best_bleu:.2f}")


In [ ]:
model = TransformerModel(
    src_vocab=len(de_tokenizer),
    tgt_vocab=len(en_tokenizer),
    d_model=384,
    nhead=6,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    max_len=80
)

In [ ]:
model = model.to(DEVICE)

In [ ]:
train_model(model, train_loader, val_loader, de_tokenizer, en_tokenizer, val_de, val_en, epochs=15, comet_experiment=comet_experiment)


In [ ]:
def translate_and_save(model, src_sentences, src_tokenizer, tgt_tokenizer, device, filename="translate.en", batch_size=128, comet_experiment=None):
    model.eval()
    results = []

    for i in tqdm(range(0, len(src_sentences), batch_size)):
        batch = src_sentences[i:i + batch_size]

        translated_batch = batch_translate(
            model, batch, src_tokenizer, tgt_tokenizer, device,
            batch_size=batch_size, max_len=100
        )
        results.extend(translated_batch)

    with open(filename, "w", encoding="utf-8") as f:
        for line in results:
            f.write(line + "\n")

    if comet_experiment is not None:
        comet_experiment.log_asset(filename, file_name=filename)
        sample_rows = [[src_sentences[i], results[i]] for i in range(min(50, len(results)))]
        comet_experiment.log_table("translation_samples.json", tabular_data=sample_rows, headers=["source_de", "pred_en"])
        comet_experiment.log_metric("translations_count", len(results))

        comet_experiment.log_asset(filename)

    print(f"Well done: {filename}")


In [ ]:
translate_and_save(
    model=model,
    src_sentences=test_de,
    src_tokenizer=de_tokenizer,
    tgt_tokenizer=en_tokenizer,
    device=DEVICE,
    filename="translate.en",
    batch_size=128,
    comet_experiment=comet_experiment
)

comet_experiment.end()